In [1]:
# 0. Establish API key

import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

#OPENAI_API_KEY

#ANTHROPIC_API_KEY

if GEMINI_API_KEY is None:
    raise RuntimeError("GEMINI_API_KEY not found in environment.")

In [2]:
# 1. Run config specs

run_config = {
  "run_id": None,

  "feature_name": "DtExp", # <-- set

  "model_provider": "gemini", # <-- set
  "model_name": "models/gemini-2.5-flash", # <-- set

  "input_csv_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/input/ntb_input.csv",     # <-- set
  "cri_json_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/DtExp.json",     # <-- set

  "data": {
      "max_tokens": None,       # None = all
      "start_index": 0,
      "shuffle": False,
      "random_seed": 0
  },

  "permutation": {
    "type": "full",            # full | ablation | allation
    "component": None
  },

  "prompt": {
    "version": "v1",
    "system_template_hash": None,
    "assembled_prompt_hash": None
  },

  "inference": {
    "temperature": 0.0,
    "max_tokens": 1024,
    "seed": None,
    "rate_limit_sleep_s": 1.0,
    "max_retries": 5
  },

  "io": {
    "output_dir": None,
    "jsonl_output_path": None,
    "csv_output_path": None
  },

  "notes": ""
}

In [3]:
# Verify model ID

#from google import genai
#client = genai.Client(api_key=GEMINI_API_KEY)

#models = client.models.list()
## show the first ~50 ids so we can spot the exact naming pattern your account sees
#print([m.name for m in models][:50])

In [4]:
#from google import genai

#client = genai.Client(api_key=GEMINI_API_KEY)

#response = client.models.generate_content(
#    model=run_config["model_name"],
#    contents="Respond with exactly the word: ALIVE"
#)

#print(response.text)

In [5]:
# 2. Auto-generate run_id, create output directory, and write config to disk

import json
from datetime import datetime, timezone
import uuid

# ---- 1. Generate run_id ----
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
short_uuid = str(uuid.uuid4())[:8]
run_id = f"{timestamp}_{short_uuid}"

run_config["run_id"] = run_id

# ---- 2. Define output directory structure ----
base_output_dir = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp"
run_output_dir = os.path.join(base_output_dir, run_id)

os.makedirs(run_output_dir, exist_ok=True)

run_config["io"]["output_dir"] = run_output_dir

# IMPORTANT:
# We will set jsonl_output_path and csv_output_path
# AFTER we know which tokens are being processed.

run_config["io"]["jsonl_output_path"] = None
run_config["io"]["csv_output_path"] = None

# ---- 3. Write config to disk immediately ----
config_path = os.path.join(run_output_dir, "run_config.json")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print(f"Run initialized: {run_id}")
print(f"Config written to: {config_path}")

Run initialized: 20260224T203259Z_62c91d38
Config written to: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T203259Z_62c91d38/run_config.json


In [6]:
# 3. Load CRI json file and implement permutation (full/ablation/allation)

from copy import deepcopy

CRI_COMPONENTS = [
    "intuition",
    "set_to_1_if",
    "set_to_0_if",
    "edge_cases",
    "examples_positive",
    "examples_negative",
]

def load_cri(cri_json_path: str) -> dict:
    with open(cri_json_path, "r", encoding="utf-8") as f:
        cri = json.load(f)
    return cri

def apply_permutation(cri: dict, permutation: dict) -> dict:
    """
    Returns a CRI dict containing ONLY the active components for this run.
    - full: all components kept
    - ablation: drop exactly one named component
    - allation: keep exactly one named component
    """
    ptype = permutation["type"]
    component = permutation.get("component")

    # Defensive copy
    cri_active = deepcopy(cri)

    if ptype == "full":
        # Ensure we only keep the known components (avoids stray metadata leaking into prompt)
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k in cri_active}

    if component not in CRI_COMPONENTS:
        raise ValueError(f"Invalid component '{component}'. Must be one of: {CRI_COMPONENTS}")

    if ptype == "ablation":
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k != component and k in cri_active}

    if ptype == "allation":
        return {component: cri_active.get(component)}

    raise ValueError(f"Invalid permutation type '{ptype}'. Must be: full | ablation | allation")

# ---- Retrieve config specs ----
cri = load_cri(run_config["cri_json_path"])
cri_active = apply_permutation(cri, run_config["permutation"])

print("Active CRI components:", list(cri_active.keys()))

bad = {k: v for k, v in cri_active.items() if v is None}
print("None-valued CRI fields:", bad)

Active CRI components: ['intuition', 'set_to_1_if', 'set_to_0_if', 'edge_cases', 'examples_positive', 'examples_negative']
None-valued CRI fields: {}


In [7]:
# 4. Assemble a deterministic prompt string + write its hash into run_config

import hashlib

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

PROMPT_TEMPLATE_V1 = """\
You are a trained linguist performing binary semantic feature annotation.

TASK:
Decide whether the TARGET GRAM instantiates the feature: {feature_name}.

EVIDENCE RULES:
- Use only the TARGET GRAM and the FULL TEXT provided.
- Follow the Constrained Reasoning Instructions (CRIs) exactly as written.
- Do not assume missing context or world knowledge beyond what is in the FULL TEXT.

META-INSTRUCTIONS (how to use the CRIs):
{meta_block}

CRIs:
{cri_block}

INPUT:
TokenID: {TokenID}
Target: {Target}
FullText: {FullText}

OUTPUT (STRICT JSON ONLY; no extra text):
{{
  "TokenID": "{TokenID}",
  "annotation": 0 or 1,
  "confidence": "high" or "medium" or "low",
  "explanation": "one sentence"
}}
"""

def render_meta_block(cri_active: dict) -> str:
    lines = []

    # Always present
    lines.append("Treat each CRI component as a different kind of evidence. Follow them as written.")

    # Only if set_to_1_if exists
    if "set_to_1_if" in cri_active:
        lines.append("Assign 1 if at least one 'set_to_1_if' condition applies.")

    # Only if set_to_0_if exists
    if "set_to_0_if" in cri_active:
        lines.append(
            "If any 'set_to_0_if' condition applies, assign 0 (this vetoes 'set_to_1_if'), "
            "unless an 'edge_cases' instruction explicitly overrides."
        )

    return "- " + "\n- ".join(lines)

def render_cri_block(cri_active: dict) -> str:
    parts = []
    for key in CRI_COMPONENTS:
        if key in cri_active and cri_active[key] is not None:
            val = cri_active[key]
            if isinstance(val, list):
                val = "\n".join(f"- {x}" for x in val)
            parts.append(f"[{key}]\n{val}")
    return "\n\n".join(parts)

def build_prompt(feature_name: str, cri_active: dict, token: dict) -> str:
    cri_block = render_cri_block(cri_active)
    meta_block = render_meta_block(cri_active)

    return PROMPT_TEMPLATE_V1.format(
        feature_name=feature_name,
        cri_block=cri_block,
        meta_block=meta_block,
        TokenID=token["TokenID"],
        Target=token["Target"],
        FullText=token["FullText"],
    )

# ---- store template hash in config ----
run_config["prompt"]["version"] = "v1"
run_config["prompt"]["system_template_hash"] = sha256_text(PROMPT_TEMPLATE_V1)

print("Template hash:", run_config["prompt"]["system_template_hash"])

Template hash: a162a52f2df71cfe965f20df69fa760eb7aaf1f55cd929401825f456846ab44a


In [8]:
# 5. Load input CSV, apply slice, set output filenames, and render one dry-run prompt

import pandas as pd

# ---- Load CSV once ----
df = pd.read_csv(run_config["input_csv_path"])

required_columns = {"TokenID", "Target", "FullText"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---- Apply run_config data slicing deterministically ----
data_cfg = run_config["data"]

df_work = df
if data_cfg.get("shuffle", False):
    df_work = df_work.sample(frac=1, random_state=data_cfg.get("random_seed", 0)).reset_index(drop=True)

start = int(data_cfg.get("start_index", 0))
max_tokens = data_cfg.get("max_tokens", None)

df_slice = df_work.iloc[start:] if max_tokens is None else df_work.iloc[start:start + int(max_tokens)]
tokens = df_slice.to_dict(orient="records")

print(f"Total tokens in file: {len(df)}")
print(f"Tokens selected for this run: {len(tokens)}")

if not tokens:
    raise RuntimeError("No tokens selected for this run (tokens list is empty).")

# ---- Set output filenames: FEAT_FIRSTTOKEN_LASTTOKEN_YYYYMMDD ----
feat = run_config["feature_name"]
first_tok = str(tokens[0]["TokenID"])
last_tok = str(tokens[-1]["TokenID"])
date_str = datetime.now(timezone.utc).strftime("%Y%m%d")

base = f"{feat}_{first_tok}_{last_tok}_{date_str}"

run_output_dir = run_config["io"]["output_dir"]
run_config["io"]["jsonl_output_path"] = os.path.join(run_output_dir, f"{base}.jsonl")
run_config["io"]["csv_output_path"] = os.path.join(run_output_dir, f"{base}.csv")

print("JSONL path:", run_config["io"]["jsonl_output_path"])
print("CSV path:  ", run_config["io"]["csv_output_path"])

# ---- Update config on disk so it matches the computed output paths ----
config_path = os.path.join(run_output_dir, "run_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ---- Dry-run: build and inspect ONE prompt ----
first_token = tokens[0]
prompt0 = build_prompt(run_config["feature_name"], cri_active, first_token)

print("\n--- PROMPT (first token) ---\n")
print(prompt0[:2000])  # truncate so you don't flood the notebook
print("\n--- END PROMPT PREVIEW ---\n")

# store an assembled prompt hash for this first instance (useful debugging)
run_config["prompt"]["assembled_prompt_hash"] = sha256_text(prompt0)
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

Total tokens in file: 594
Tokens selected for this run: 594
JSONL path: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T203259Z_62c91d38/DtExp_40001018_45007006a_20260224.jsonl
CSV path:   /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T203259Z_62c91d38/DtExp_40001018_45007006a_20260224.csv

--- PROMPT (first token) ---

You are a trained linguist performing binary semantic feature annotation.

TASK:
Decide whether the TARGET GRAM instantiates the feature: DtExp.

EVIDENCE RULES:
- Use only the TARGET GRAM and the FULL TEXT provided.
- Follow the Constrained Reasoning Instructions (CRIs) exactly as written.
- Do not assume missing context or world knowledge beyond what is in the FULL TEXT.

META-INSTRUCTIONS (how to use the CRIs):
- Treat each CRI component as a different kind of evidence. F

In [9]:
# 6. Strict response parser + validator (brace-slice extraction)

def parse_model_response(raw_text: str) -> dict:
    if raw_text is None:
        raise ValueError("Model output is empty (None).")

    text = raw_text.strip()

    # If it starts with a code fence, drop the first fence line and the last fence line if present
    if text.startswith("```"):
        lines = text.splitlines()
        # drop first line (``` or ```json)
        lines = lines[1:]
        # drop last line if it's a closing fence
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"Could not find JSON braces in model output. Starts with: {text[:120]!r}")

    json_text = text[start:end+1].strip()

    try:
        data = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Model output is not valid JSON. JSONDecodeError: {e}. Extracted starts with: {json_text[:120]!r}"
        )

    required_keys = {"TokenID", "annotation", "confidence", "explanation"}
    missing = required_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing required JSON keys: {missing}. Got keys: {set(data.keys())}")

    # Coerce TokenID to string (models sometimes return ints)
    data["TokenID"] = str(data["TokenID"])

    # Coerce annotation if it comes back as "0"/"1"
    if isinstance(data["annotation"], str) and data["annotation"].strip() in ("0", "1"):
        data["annotation"] = int(data["annotation"].strip())

    if data["annotation"] not in (0, 1):
        raise ValueError("Annotation must be 0 or 1.")

    if isinstance(data["confidence"], str):
        data["confidence"] = data["confidence"].lower().strip()

    if data["confidence"] not in ("high", "medium", "low"):
        raise ValueError("Confidence must be 'high', 'medium', or 'low'.")

    if not isinstance(data["explanation"], str):
        raise ValueError("Explanation must be a string.")

    # Optional: if you want to enforce *only* these outputs downstream
    return {k: data[k] for k in required_keys}

In [10]:
# 7. Model call wrapper (Gemini + provider dispatcher)

import time
from google import genai
from google.genai import errors

# ---- Gemini client ----
client = genai.Client(api_key=GEMINI_API_KEY)

def call_gemini(prompt: str) -> str:
    max_retries = int(run_config["inference"].get("max_retries", 3))
    base_sleep = float(run_config["inference"].get("rate_limit_sleep_s", 1.0))

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=run_config["model_name"],
                contents=prompt,
                config={
                    "temperature": run_config["inference"]["temperature"],
                    "max_output_tokens": run_config["inference"]["max_tokens"],
                    "response_mime_type": "application/json",
                    "response_schema": {
                        "type": "object",
                        "properties": {
                            "TokenID": {"type": "string"},
                            "annotation": {"type": "string", "enum": ["0", "1"]},
                            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                            "explanation": {"type": "string"},
                        },
                        "required": ["TokenID", "annotation", "confidence", "explanation"],
                    },
                },
            )

            # Robustly join all text parts (response.text can be partial)
            parts = []
            for cand in getattr(response, "candidates", []) or []:
                content = getattr(cand, "content", None)
                for part in getattr(content, "parts", []) or []:
                    t = getattr(part, "text", None)
                    if t:
                        parts.append(t)

            return "".join(parts) if parts else (response.text or "")

        except errors.ClientError as e:
            last_err = e
            # 429 = resource exhausted / rate limit / quota
            if getattr(e, "status_code", None) == 429 or "RESOURCE_EXHAUSTED" in str(e):
                sleep_s = max(base_sleep, 1.0) * (2 ** (attempt - 1))
                print(f"[429] Resource exhausted. Sleeping {sleep_s:.1f}s then retrying ({attempt}/{max_retries})...")
                time.sleep(sleep_s)
                continue
            raise

    raise RuntimeError(f"Gemini call failed after {max_retries} retries. Last error: {last_err}")

# ---- Provider dispatcher (future-proof) ----
def call_model(prompt: str) -> str:
    provider = run_config["model_provider"]

    if provider == "gemini":
        return call_gemini(prompt)

    # We will implement these in the next thread
    elif provider == "openai":
        raise NotImplementedError("OpenAI wrapper not yet implemented.")
    elif provider == "anthropic":
        raise NotImplementedError("Anthropic wrapper not yet implemented.")

    else:
        raise ValueError(f"Unknown model_provider: {provider}")

In [11]:
# 8. Single-token live test

#first_token = tokens[0]

#prompt0 = build_prompt(
#    run_config["feature_name"],
#    cri_active,
#    first_token
#)

#print("Sending prompt for TokenID:", first_token["TokenID"])

#raw_output = call_model(prompt0)

#print("\n--- RAW MODEL OUTPUT ---\n")
#print(raw_output)
#print("\n--- END RAW OUTPUT ---\n")

In [12]:
# 9. Batch annotation loop (chunked)

import time
from pathlib import Path

batch_size = 10  # <-- change to 50 later if you want
sleep_s = run_config["inference"]["rate_limit_sleep_s"]

# Base directory for this run (already set earlier in your pipeline)
out_dir = Path(run_config["io"]["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Writing outputs under: {out_dir}")
print(f"Total tokens in memory: {len(tokens)}")
print(f"Chunk size: {batch_size}")

total_processed = 0
total_failures = 0

for batch_start in range(0, len(tokens), batch_size):
    batch = tokens[batch_start: batch_start + batch_size]
    first_id = batch[0]["TokenID"]
    last_id  = batch[-1]["TokenID"]

    # Write ONE jsonl per chunk so you can resume / inspect easily
    date_str = run_config["run_id"][:8]  # run_id starts with YYYYMMDD
    jsonl_path = out_dir / f'{run_config["feature_name"]}_{first_id}_{last_id}_{date_str}.jsonl'

    processed = 0
    failures = 0

    print(f"\nBatch {batch_start//batch_size + 1}: {first_id} → {last_id}")
    print(f"JSONL: {jsonl_path}")

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for token in batch:
            token_id = token["TokenID"]
            prompt = build_prompt(run_config["feature_name"], cri_active, token)

            try:
                raw = call_model(prompt)
                parsed = parse_model_response(raw)

                # Provenance (matching your run_config keys)
                parsed["run_id"] = run_config["run_id"]
                parsed["model_provider"] = run_config["model_provider"]
                parsed["model_name"] = run_config["model_name"]
                parsed["feature_name"] = run_config["feature_name"]
                parsed["permutation_type"] = run_config["permutation"]["type"]
                parsed["permutation_component"] = run_config["permutation"].get("component")

                f.write(json.dumps(parsed) + "\n")
                processed += 1
                total_processed += 1

            except Exception as e:
                error_record = {
                    "TokenID": token_id,
                    "annotation": None,
                    "confidence": None,
                    "explanation": None,
                    "error": str(e),
                }
                f.write(json.dumps(error_record) + "\n")
                failures += 1
                total_failures += 1

            time.sleep(sleep_s)

    print(f"Batch done. Processed: {processed}, Failures: {failures}")

print("\nALL DONE")
print("Total processed:", total_processed)
print("Total failures:", total_failures)

Writing outputs under: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T203259Z_62c91d38
Total tokens in memory: 594
Chunk size: 10

Batch 1: 40001018 → 40006005
JSONL: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T203259Z_62c91d38/DtExp_40001018_40006005_20260224.jsonl
Batch done. Processed: 0, Failures: 10

Batch 2: 40006016 → 40011012a
JSONL: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T203259Z_62c91d38/DtExp_40006016_40011012a_20260224.jsonl


KeyboardInterrupt: 

In [ ]:
import json
with open("/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp/20260224T195117Z_742a5f01/DtExp_40001018_40006005.jsonl") as f:
    print(json.loads(f.readline())["error"])

In [ ]:
# 10. Merge JSONL results back into CSV

import pandas as pd
import json

jsonl_path = run_config["io"]["jsonl_output_path"]
csv_output_path = run_config["io"]["csv_output_path"]

if jsonl_path is None or csv_output_path is None:
    raise RuntimeError("Output paths not set. Check Cell 5.")

# Reconstruct the exact input slice from tokens (no re-slicing logic)
df_slice = pd.DataFrame(tokens)

# Add run metadata to every row (redundant with JSONL, but very useful)
df_slice["run_id"] = run_config["run_id"]
df_slice["model_provider"] = run_config["model_provider"]
df_slice["model_name"] = run_config["model_name"]
df_slice["permutation_type"] = run_config["permutation_type"]
df_slice["feature_name"] = run_config["feature_name"]

# Load JSONL results (one file per batch written by Cell 9)
from pathlib import Path
out_dir = Path(run_config["io"]["output_dir"])
jsonl_files = sorted(out_dir.glob(f'{run_config["feature_name"]}_*.jsonl'))
if not jsonl_files:
    raise RuntimeError(f"No JSONL files found in {out_dir}")
records = []
for jf in jsonl_files:
    with open(jf, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
print(f"Loaded {len(records)} records from {len(jsonl_files)} file(s)")

df_results = pd.DataFrame(records)

# If JSONL already includes these metadata fields, drop them to prevent *_x/*_y columns after merge
df_results = df_results.drop(
    columns=["run_id", "model_provider", "model_name", "feature_name",
             "permutation_type", "permutation_component"],
    errors="ignore"
)

# Merge on TokenID
df_merged = df_slice.merge(df_results, on="TokenID", how="left")

df_merged.to_csv(csv_output_path, index=False)

print("Merged CSV written to:", csv_output_path)